# Sprint 4 Mejora C: Descarga resumable con auto-save

**Problema del notebook anterior:** copiaba a Drive solo al final.
Si la sesión se cerraba, todo el progreso se perdía.

**Solución de este notebook:**
- Después de procesar **cada tarball**, guarda el HDF5 en Drive.
- Si la sesión se cierra, el próximo intento continúa desde el último tarball guardado.
- El notebook detecta automáticamente qué tarballs ya están en el HDF5.

**Estado actual:** HDF5 en Drive tiene las 34,999 imágenes de los 5 tarballs originales.
**Objetivo:** agregar ~70,000 imágenes más de los tarballs 006-012.

**Runtime:** CPU. Tiempo estimado: ~3-4 horas (pasivo).


## 1. Setup y verificar estado actual

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

H5_DRIVE = f'{NIH_DIR}/nih_images.h5'
H5_LOCAL = '/content/nih_images.h5'

import os, shutil, time, tarfile, io, h5py
import numpy as np
from PIL import Image

!pip install -q h5py tqdm
from tqdm import tqdm

# Verificar HDF5 en Drive (siempre existe, tiene 34,999 imgs)
size_drive = os.path.getsize(H5_DRIVE) / (1024**3)
with h5py.File(H5_DRIVE, 'r') as hf:
    n_drive = len(hf['images'])

print(f'HDF5 en Drive:  {n_drive:,} imágenes ({size_drive:.2f} GB)')
print()

# Verificar espacio disponible
import shutil as _sh
free_local = _sh.disk_usage('/content').free / (1024**3)
free_drive = _sh.disk_usage('/content/drive/MyDrive').free / (1024**3)
print(f'Espacio /content/:          {free_local:.1f} GB')
print(f'Espacio Drive:              {free_drive:.1f} GB')
print()

# Estado objetivo
print('ESTADO OBJETIVO:')
print('  Tarballs 001-005: ✓ ya en HDF5')
for i in range(6, 13):
    print(f'  Tarball {i:03d}:      ← pendiente')


## 2. Copiar HDF5 de Drive a /content/

Copiamos el HDF5 actual (con lo que ya está guardado) a disco local.
Así las operaciones de escritura serán rápidas y sin errores de Drive.


In [ ]:
if os.path.exists(H5_LOCAL):
    local_size = os.path.getsize(H5_LOCAL) / (1024**3)
    with h5py.File(H5_LOCAL, 'r') as hf:
        n_local = len(hf['images'])
    # Si el local tiene MÁS imágenes que Drive, ya progresamos en esta sesión
    if n_local >= n_drive:
        print(f'✓ HDF5 local más reciente ({n_local:,} imgs). Usando el local.')
    else:
        print(f'HDF5 local desactualizado ({n_local:,} vs {n_drive:,} en Drive).')
        print(f'Copiando desde Drive...')
        t0 = time.time()
        shutil.copy2(H5_DRIVE, H5_LOCAL)
        elapsed = time.time() - t0
        with h5py.File(H5_LOCAL, 'r') as hf:
            n_local = len(hf['images'])
        print(f'✓ Copiado en {elapsed/60:.1f} min | {n_local:,} imágenes')
else:
    print(f'Copiando HDF5 de Drive a /content/ ({size_drive:.2f} GB)...')
    t0 = time.time()
    shutil.copy2(H5_DRIVE, H5_LOCAL)
    elapsed = time.time() - t0
    with h5py.File(H5_LOCAL, 'r') as hf:
        n_local = len(hf['images'])
    print(f'✓ Copiado en {elapsed/60:.1f} min | {n_local:,} imágenes')

# Ver qué imágenes ya están
with h5py.File(H5_LOCAL, 'r') as hf:
    existing_in_h5 = set(hf['images'].keys())
print(f'\nImágenes en HDF5 local: {len(existing_in_h5):,}')


## 3. Descargar tarballs 006-012 con auto-save

**Auto-save activado:** después de cada tarball, el HDF5 se copia a Drive.
Si la sesión se cierra, el próximo intento retoma desde el último guardado.

Proceso por tarball:
1. Descarga a `/content/` (~3.7 GB).
2. Lee imágenes en memoria y las agrega al HDF5 local.
3. **Borra tarball** de `/content/` para liberar espacio.
4. **Copia HDF5 a Drive** (auto-save del progreso).


In [ ]:
BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'
TARBALLS = [f'images_{i:03d}.tar.gz' for i in range(6, 13)]

print('Tarballs a procesar: 006 a 012')
print(f'Ya en HDF5: {len(existing_in_h5):,} imágenes')
print()

def save_to_drive(h5_local, h5_drive, label=''):
    """Copia HDF5 local a Drive y reporta el progreso."""
    print(f'  💾 Auto-save a Drive{" ("+label+")" if label else ""}...', end=' ')
    t0 = time.time()
    shutil.copy2(h5_local, h5_drive)
    elapsed = time.time() - t0
    with h5py.File(h5_drive, 'r') as hf:
        n = len(hf['images'])
    size_gb = os.path.getsize(h5_drive) / (1024**3)
    print(f'✓ {n:,} imgs | {size_gb:.2f} GB | {elapsed/60:.1f} min')


total_added = 0
t_global = time.time()

for tb_name in TARBALLS:
    print(f'\n{"="*60}')
    print(f'TARBALL: {tb_name}')
    print(f'{"="*60}')

    url       = f'{BASE_URL}/{tb_name}'
    tar_local = f'/content/{tb_name}'

    # ── Descarga (con resume automático si existe parcialmente) ──
    if os.path.exists(tar_local) and os.path.getsize(tar_local) > 3_500_000_000 * 0.95:
        size_gb = os.path.getsize(tar_local) / (1024**3)
        print(f'✓ Ya descargado ({size_gb:.2f} GB). Saltando descarga.')
    else:
        print(f'Descargando desde Archive.org...')
        ret = os.system(
            f'wget -c --progress=bar:force:noscroll -O "{tar_local}" "{url}"'
        )
        if ret != 0 or not os.path.exists(tar_local):
            print(f'✗ Error en descarga. Saltando este tarball.')
            continue

    size_gb = os.path.getsize(tar_local) / (1024**3)
    print(f'Tamaño: {size_gb:.2f} GB')

    # ── Verificar integridad ─────────────────────────────────────
    try:
        with tarfile.open(tar_local, 'r:gz') as tar:
            members = [m for m in tar.getmembers()
                       if m.isfile() and m.name.endswith('.png')]
        new_members_count = sum(
            1 for m in members
            if os.path.basename(m.name) not in existing_in_h5
        )
        print(f'Archivos en tarball: {len(members):,} | '
              f'Nuevos para HDF5: {new_members_count:,}')
    except Exception as e:
        print(f'✗ Tarball corrupto: {e}. Saltando.')
        os.remove(tar_local)
        continue

    if new_members_count == 0:
        print(f'✓ Todas las imágenes ya están en HDF5. Saltando.')
        os.remove(tar_local)
        continue

    # ── Agregar imágenes al HDF5 ─────────────────────────────────
    saved, failed = 0, 0
    t0 = time.time()

    with tarfile.open(tar_local, 'r:gz') as tar,          h5py.File(H5_LOCAL, 'a') as hf:

        grp = hf.require_group('images')
        members_to_add = [
            m for m in members
            if os.path.basename(m.name) not in existing_in_h5
        ]

        for m in tqdm(members_to_add, desc=tb_name, unit='img'):
            img_name = os.path.basename(m.name)
            try:
                f = tar.extractfile(m)
                if f is None:
                    continue
                raw = f.read()
                with Image.open(io.BytesIO(raw)) as pil_img:
                    arr = np.array(
                        pil_img.convert('L').resize((256, 256), Image.LANCZOS),
                        dtype=np.uint8
                    )
                grp.create_dataset(img_name, data=arr,
                                   compression='gzip', compression_opts=4)
                existing_in_h5.add(img_name)
                saved += 1
            except Exception:
                failed += 1

    elapsed = time.time() - t0
    total_added += saved
    print(f'✓ Agregadas: {saved:,} | Fallidas: {failed} | '
          f'Tiempo: {elapsed/60:.1f} min')

    # ── Borrar tarball para liberar espacio ──────────────────────
    os.remove(tar_local)
    free_now = _sh.disk_usage('/content').free / (1024**3)
    print(f'  Tarball borrado | Espacio libre: {free_now:.1f} GB')

    # ── AUTO-SAVE: copiar HDF5 a Drive ───────────────────────────
    with h5py.File(H5_LOCAL, 'r') as hf:
        n_now = len(hf['images'])
    save_to_drive(H5_LOCAL, H5_DRIVE, label=f'después de {tb_name}')
    print(f'  HDF5 total: {n_now:,} imágenes')

# ── Resumen final ─────────────────────────────────────────────────
elapsed_total = time.time() - t_global
with h5py.File(H5_LOCAL, 'r') as hf:
    n_final = len(hf['images'])

print()
print('='*60)
print(f'PROCESO COMPLETADO en {elapsed_total/60:.0f} min')
print(f'  Imágenes originales: 34,999')
print(f'  Imágenes agregadas:  {total_added:,}')
print(f'  Total en HDF5:       {n_final:,}')
print(f'  HDF5 guardado en Drive: {H5_DRIVE}')
print('='*60)


## 4. Actualizar CSV y ver distribución final

In [ ]:
import pandas as pd

df_full = pd.read_csv(f'{NIH_DIR}/Data_Entry_2017.csv')
with h5py.File(H5_LOCAL, 'r') as hf:
    in_h5 = set(hf['images'].keys())

df_subset = df_full[df_full['Image Index'].isin(in_h5)].copy()
df_subset.to_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv', index=False)
print(f'CSV actualizado: {len(df_subset):,} filas')
print()

# Distribución de las 4 clases
print('DISTRIBUCIÓN FINAL (12 tarballs):')
print('='*55)
CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}

def assign_label(finding):
    for kw, lbl in [('Infiltration',3),('Effusion',2),('Cardiomegaly',1)]:
        if kw in finding: return lbl
    return 0 if finding == 'No Finding' else -1

df_subset['label'] = df_subset['Finding Labels'].apply(assign_label)
df_valid = df_subset[df_subset['label'] >= 0]
pat_labels = df_valid.groupby('Patient ID')['label'].max()

for lbl, name in CLASS_NAMES.items():
    n_imgs = (df_subset['label'] == lbl).sum()
    n_pats = (pat_labels == lbl).sum()
    print(f'{name:<20}: {n_imgs:>8,} imágenes | {n_pats:>6,} pacientes únicos')

ceiling = min((pat_labels == l).sum() for l in [1,2,3])
nf_pats = min((pat_labels == 0).sum(), ceiling * 2)
total_pats = ceiling * 3 + nf_pats

print()
print(f'Techo (clase más pequeña): {ceiling:,} pacientes')
print(f'Dataset balanceado estimado:')
print(f'  {ceiling:,} × 3 positivas + {nf_pats:,} NF = {total_pats:,} pacientes')
print(f'  Train (~todas imgs/paciente × ~4): ~{total_pats*4:,} imágenes')
print()

# Comparativa con 5 tarballs
print('Comparativa 5 vs 12 tarballs:')
print(f'  Techo (Cardiomegaly): 260 → {ceiling:,} pacientes (+{ceiling/260:.0f}x)')
print(f'  Train estimado: 3,900 → ~{total_pats*4:,} imágenes')


## 5. Siguiente paso

Con el HDF5 ampliado guardado en Drive, abre `S4_04_retrain_full.ipynb`
en una nueva sesión con **T4 GPU** para reentrenar el CNN-ViT con A+B+C.

Si la sesión se cierra durante esta descarga, simplemente:
1. Reabre este notebook.
2. Ejecuta desde la celda 1.
3. El notebook detecta el progreso guardado en Drive y continúa.
